# 📊 Day 3, Topics 1 & 2 — Handling Missing Data: Detection & Dropping

Missing data is the most common data quality problem. Before you can fix it, you must **find it**.

## Why missing data matters
- Most ML algorithms can't handle `NaN` — they will crash or produce wrong results
- Statistics computed on incomplete data can be biased
- Missing patterns themselves can carry information (e.g., wealthy passengers often had no Cabin listed)

## Pandas' missing value sentinel: `NaN`
Pandas uses `float NaN` (Not a Number) to represent missing values in numeric columns, and `None` or `NaN` in object columns. Both are caught by `.isna()`.

## Day 3 Overview
| Topic | Subject |
|-------|---------|
| 1 | Detecting missing values |
| 2 | Dropping rows/columns with missing values |
| 3 | Filling missing values |
| 4 | Finding and removing duplicates |
| 5 | Data type conversion |
| 6 | String cleaning |
| 7 | Renaming columns |
| 8 | Replacing values |
| 9 | Binning continuous data |
| 10 | Creating dummy variables |


Day 3, Topic 1: Handling Missing Data – Detection

## Topic 1 — Detecting Missing Data

### The detection toolkit

| Method | Returns | Use for |
|--------|---------|---------|
| `df.isna()` | Boolean DataFrame — True where NaN | Full missing map |
| `df.isna().sum()` | Count of NaN per column | Quick column overview |
| `df.isna().sum() / len(df)` | Fraction missing per column | Percentage missing |
| `df.notna()` | Boolean DataFrame — True where NOT NaN | Inverse mask |
| `df.info()` | Prints Non-Null counts | Combined dtype + missing view |
| `df.isna().any(axis=1)` | Boolean Series — True if row has any NaN | Row-level check |


In [2]:
import pandas as pd

df = pd.read_csv('titanic.csv')

### `.isna()` — Boolean Missing Map
`df.isna()` returns a DataFrame of the same shape where every cell is `True` (missing) or `False` (present).  
Chaining `.sum()` collapses it to counts per column.


In [3]:
#.isna() and .isnull()
mask = df.isna()

mask.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,False,False,False,False,False,False,False,False,False,False,True,False
1,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,True,False
3,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,True,False


In [4]:
df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

### `.notna()` — The Inverse
`df.notna().sum()` counts **non-missing** values per column.  
This is the same as `len(df) - df.isna().sum()`, but more direct.


In [5]:
#.notna() – The Inverse

df.notna().sum()

PassengerId    891
Survived       891
Pclass         891
Name           891
Sex            891
Age            714
SibSp          891
Parch          891
Ticket         891
Fare           891
Cabin          204
Embarked       889
dtype: int64

### `.info()` — Fastest Overall View
`.info()` prints `Non-Null Count` for every column in one call.  
The gap between total rows (891) and non-null count immediately reveals missing data:  
- `Age: 714 non-null` → 177 missing  
- `Cabin: 204 non-null` → 687 missing (77%!)


In [6]:
#.info() – The Quickest Overview
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


---
### 📝 Practice — Missing Data Detection


In [8]:
#Practice Activity: Missing Data Detection
import pandas as pd
df = pd.read_csv("titanic.csv")

#### Task 1 — Column Missing Counts
`df.isna().sum()` — one number per column. Sort with `.sort_values(ascending=False)` to rank by missingness.


In [12]:
#Task 1: Use .isna().sum() to display the count of missing values in each column.
missing_counts = df.isna().sum()
missing_counts

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

#### Task 2 — Confirm with `.info()`
`.info()` cross-checks Task 1. Columns with > 100 missing: `Age` (177) and `Cabin` (687).


In [13]:
"""Task 2: Use .info() to confirm which columns have missing data. 
Write down the names of columns with more than 100 missing values."""
df.info()

high_missing = missing_counts[missing_counts > 100]
high_missing.index.tolist()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


['Age', 'Cabin']

#### Task 3 — Boolean Mask for Missing Age
`missing_age = df['Age'].isna()` — True for the 177 rows missing an age.  
Using it in `df.loc[missing_age, ['Name','Pclass']]` shows who is affected.


In [15]:
"""Task 3: Create a Boolean Series missing_age that is True for rows where Age is missing. 
Use it to display the Name and Pclass of passengers with missing ages."""

mask = df['Age'].isna()
result = df[mask]
result[['Name', 'Age']].head()
len(result)

177

#### Task 4 — Rows with Any Missing Value
`df.isna().any(axis=1)` → True if **any** column in that row is NaN.  
`.sum()` counts how many such rows exist. `axis=1` means "check across columns".


In [16]:
#Task 4: How many rows have at least one missing value?

df.isna().any(axis=1).sum()

np.int64(708)

#### Task 5 — Percentage Missing per Column
`(df.isna().sum() / len(df)) * 100` gives the percentage. Sort to find the worst offenders.  
`Cabin` is ~77% missing — a strong signal to drop it entirely.


In [17]:
#Task 5: Which column has the highest percentage of missing values? Calculate it.

missing_percents = (df.isna().sum() / len(df)) * 100
highest_col = missing_percents.idxmax()
highest_pcl = missing_percents.max()
print(f"Column '{highest_col}' has {highest_pcl:.1f}% missing values")
missing_percents.sort_values(ascending=False)

Column 'Cabin' has 77.1% missing values


Cabin          77.104377
Age            19.865320
Embarked        0.224467
PassengerId     0.000000
Survived        0.000000
Pclass          0.000000
Name            0.000000
Sex             0.000000
SibSp           0.000000
Parch           0.000000
Ticket          0.000000
Fare            0.000000
dtype: float64

In [18]:
#Day 3, Topic 2: Handling Missing Data – Dropping
import pandas as pd

df = pd.read_csv('titanic.csv')



## Topic 2 — Dropping Missing Data

### When to drop vs fill?
| Situation | Action |
|-----------|--------|
| Column > 50–70% missing | **Drop the column** — too little data to be useful |
| Column < 5% missing | **Drop those rows** — small loss |
| Column 5–50% missing | **Fill** with a statistic or model |
| Rows where ALL key cols are missing | **Drop rows** with `how='all'` |

### `.dropna()` Parameters
```python
df.dropna()                          # drop any row with ANY NaN
df.dropna(subset=['Age'])            # drop rows where Age is NaN
df.dropna(subset=['A','B'], how='all') # drop rows where BOTH A AND B are NaN
df.dropna(axis=1)                    # drop any COLUMN with ANY NaN
df.dropna(axis=1, thresh=500)        # keep columns with at least 500 non-null
df.dropna(inplace=True)              # modify df directly (no reassignment)
```


### Basic `.dropna()` — Drop Any Row with Any Missing Value
From 891 to ~183 rows — a massive loss! Dropping all rows with any NaN is almost never the right approach on real datasets.  
Always inspect what you're dropping before committing.


In [19]:
#Basic .dropna() – Drop Any Row with a Missing Value
df_clean = df.dropna()

print(df.shape)
print(df_clean.shape)

(891, 12)
(183, 12)


### `subset=` — Drop Rows Based on Specific Columns
`df.dropna(subset=['Age'])` drops only the 177 rows where `Age` is missing, keeping the rest.  
Much more targeted than dropping all rows with any NaN.


In [20]:
#Dropping Rows Based on Specific Columns (subset)
df_age_clear = df.dropna(subset=['Age'])
df_age_clear.shape

(714, 12)

In [21]:
df.dropna(subset=['Name', 'Embarked']).shape

(889, 12)

### `how='all'` — Drop Only if ALL Specified Columns are Missing
`how='all'` (vs `how='any'` default): a row is dropped only when **every** column in the subset is NaN.  
Much more permissive — you only lose rows with no useful data at all.


In [22]:
df.dropna(subset=['Age', 'Cabin', 'Embarked'], how='all').shape

(891, 12)

### `axis=1` — Drop Columns Instead of Rows
`df.dropna(axis=1)` removes any column that has even one missing value.  
On Titanic this drops `Age`, `Cabin`, and `Embarked` — leaving only fully-complete columns.


In [23]:
#Dropping Columns with Missing Values
df_drop_cols = df.dropna(axis=1)
df_drop_cols.columns.tolist()

['PassengerId',
 'Survived',
 'Pclass',
 'Name',
 'Sex',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare']

### `thresh=` — Minimum Non-Null Threshold
`thresh=500` keeps only columns with **at least 500** non-null values.  
`Cabin` (204 non-null) gets dropped; `Age` (714 non-null) is kept.  
A powerful way to automatically prune very sparse columns.


In [24]:
#The thresh Parameter – Minimum Non‑Null Values
df_thresh = df.dropna(axis=1, thresh=500)
df_thresh.columns.tolist()

['PassengerId',
 'Survived',
 'Pclass',
 'Name',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Embarked']

### `inplace=True` — Modify in Place
`inplace=True` modifies `df` directly without needing `df = df.dropna(...)`.  
⚠️ Use with care — the original is gone unless you made a copy first with `df.copy()`.


In [27]:
#In‑Place Dropping
df.dropna(subset=['Embarked'], inplace=True)


---
### 📝 Practice — Dropping Missing Data


In [28]:
#Practice Activity: Dropping Missing Data
import pandas as pd

df = pd.read_csv('titanic.csv')

#### Task 1 — Drop Rows Where Embarked is Missing
Only 2 rows have missing Embarked — a tiny, safe drop.


In [31]:
#Task 1: Drop all rows where Embarked is missing. How many rows remain?

df_clean = df.dropna(subset=['Embarked'])
len(df_clean)

889

#### Task 2 — `how='all'` on Age and Cabin
Drops rows where **both** Age AND Cabin are NaN simultaneously — fewer rows dropped than either alone.


In [33]:
#Task 2: Drop all rows where both Age AND Cabin are missing (hint: how='all' with subset). How many rows are dropped?

df_clean_1 = df.dropna(subset=['Age', 'Cabin'], how='all')
len(df_clean_1)

733

#### Task 3 — Keep Only Complete Columns
`df.dropna(axis=1)` → keeps columns with zero missing values. Lists their names.


In [36]:
"""Task 3: Create a new DataFrame that contains only columns with zero missing values. 
How many columns are kept? List their names."""

col = df.dropna(axis=1)
col.columns.tolist()
col.shape[1]


9

#### Task 4 — `thresh` to Prune Sparse Columns
`thresh=800` keeps columns with ≥ 800 non-null values. `Cabin` (204) and `Age` (714) are dropped.


In [38]:
"""Task 4: Use the thresh parameter to keep only columns that have at least 800 non‑null values. 
Which columns are dropped?"""

df_thresh = df.dropna(axis=1, thresh=800)
set(df.columns) - set(df_thresh)

{'Age', 'Cabin'}

#### Task 5 (Bonus) — Replace then Drop
Replace `Fare == 0` with `NaN` using `.replace(0, np.nan)`, then drop rows where Age or Fare is NaN.  
Demonstrates that "impossible" values should be treated as missing before dropping.


In [39]:
"""Task 5 (Bonus): Drop rows where Age is missing or Fare is 0 (impossible fare). 
First replace Fare of 0 with NaN using .replace(0, np.nan), then drop rows with missing Age or Fare. 
How many rows remain?"""
import numpy as np
df['Fare'] = df['Fare'].replace(0, np.nan)
df_cleaned = df.dropna(subset=['Age', 'Fare'])
len(df_cleaned)

707